# 🌍 Track 2: Population Health & Health Equity Analytics
## UVic Healthcare AI Hackathon — March 27-28, 2026

Welcome! This notebook will walk you through:
1. Loading BC community health data, CIHI wait times, and opioid surveillance data
2. Mapping health indicators across BC communities
3. Analyzing correlations between social determinants and health outcomes
4. Building a wait time forecasting model
5. Ideas for extending your project into a health equity tool

**Context:** British Columbia has a publicly funded healthcare system (MSP). 
Island Health (Vancouver Island Health Authority) serves our region. 
This data reflects real patterns in BC's health landscape.

**Time estimate:** 30-45 minutes to work through, then build something amazing.

In [ ]:
# Setup — run this cell first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Data directory — adjust if you moved things around
DATA_DIR = Path(".")
# If running from repo root instead of inside the track folder:
if not (DATA_DIR / "bc-community-profiles").exists():
    DATA_DIR = Path("track-2-population-health")

print("✅ Setup complete!")
print(f"📁 Data directory: {DATA_DIR.resolve()}")

---
## 📥 Load Data

In [ ]:
# Load all population health datasets
bc_health = pd.read_csv(DATA_DIR / "bc-community-profiles" / "bc_health_indicators.csv")
wait_times = pd.read_csv(DATA_DIR / "cihi-wait-times" / "wait_times_mock.csv")
opioid_data = pd.read_csv(DATA_DIR / "opioid-surveillance" / "opioid_harms_mock.csv")

print(f"BC Health Indicators: {len(bc_health)} communities, {len(bc_health.columns)} indicators")
print(f"Wait Times: {len(wait_times):,} records ({wait_times['year'].min()}-{wait_times['year'].max()})")
print(f"Opioid Surveillance: {len(opioid_data):,} records")

In [ ]:
bc_health.head(10)

In [ ]:
# Focus on Island Health communities
island_health = bc_health[bc_health['health_authority'] == 'Island Health'].copy()
print(f"Island Health communities: {len(island_health)}")
island_health[[
    'chsa_name', 'population', 'median_age', 'life_expectancy',
    'diabetes_prevalence', 'opioid_overdose_rate'
]].sort_values('population', ascending=False)

---
## 🗺️ Mapping Health Across BC Communities

Let's visualize how health indicators vary across communities and health authorities in British Columbia.

In [ ]:
# Population by Island Health community
fig, ax = plt.subplots(figsize=(12, 7))
ih_sorted = island_health.sort_values('population', ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(ih_sorted)))
ax.barh(ih_sorted['chsa_name'], ih_sorted['population'], color=colors)
ax.set_xlabel('Population')
ax.set_title('Population by Community — Island Health Authority')
for i, (val, name) in enumerate(zip(ih_sorted['population'], ih_sorted['chsa_name'])):
    ax.text(val + 500, i, f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Median age vs life expectancy — colored by health authority
fig, ax = plt.subplots(figsize=(12, 7))
ha_colors = {
    'Island Health': '#2196F3',
    'Vancouver Coastal': '#4CAF50',
    'Fraser': '#FF9800',
    'Interior': '#9C27B0',
    'Northern': '#F44336',
}
for ha, color in ha_colors.items():
    subset = bc_health[bc_health['health_authority'] == ha]
    ax.scatter(subset['median_age'], subset['life_expectancy'],
               c=color, label=ha, s=subset['population'] / 500,
               alpha=0.7, edgecolors='white', linewidth=0.5)
ax.set_xlabel('Median Age (years)')
ax.set_ylabel('Life Expectancy (years)')
ax.set_title('Median Age vs. Life Expectancy by BC Community\n(bubble size = population)')
ax.legend(title='Health Authority', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 communities by diabetes prevalence
top_diabetes = bc_health.nlargest(20, 'diabetes_prevalence')

fig, ax = plt.subplots(figsize=(12, 8))
bar_colors = [ha_colors.get(ha, '#888') for ha in top_diabetes['health_authority']]
bars = ax.barh(
    top_diabetes['chsa_name'],
    top_diabetes['diabetes_prevalence'],
    color=bar_colors
)
ax.set_xlabel('Diabetes Prevalence (%)')
ax.set_title('Top 20 BC Communities by Diabetes Prevalence')
ax.axvline(bc_health['diabetes_prevalence'].mean(), color='red',
           linestyle='--', alpha=0.7, label=f"BC Average ({bc_health['diabetes_prevalence'].mean():.1f}%)")
ax.legend(fontsize=10)
# Add HA legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=ha) for ha, c in ha_colors.items()]
ax.legend(handles=legend_patches, title='Health Authority', loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Key indicators by health authority — grouped bar chart
indicator_cols = ['diabetes_prevalence', 'pct_obese', 'pct_smokers', 'opioid_overdose_rate']
indicator_labels = ['Diabetes\nPrevalence (%)', 'Obesity\nRate (%)', 'Smoking\nRate (%)', 'Opioid Overdose\nRate (per 100K)']

ha_means = bc_health.groupby('health_authority')[indicator_cols].mean()

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(indicator_cols))
width = 0.15
for i, (ha, row) in enumerate(ha_means.iterrows()):
    offset = (i - len(ha_means) / 2 + 0.5) * width
    ax.bar(x + offset, row.values, width, label=ha, color=ha_colors.get(ha, '#888'))

ax.set_xticks(x)
ax.set_xticklabels(indicator_labels, fontsize=11)
ax.set_ylabel('Rate / Prevalence')
ax.set_title('Key Health Indicators by Health Authority')
ax.legend(title='Health Authority', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of all numeric health indicators
numeric_cols = bc_health.select_dtypes(include=[np.number]).columns.tolist()
# Remove the code column
numeric_cols = [c for c in numeric_cols if c != 'chsa_code']

corr = bc_health[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix: BC Community Health Indicators', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

---
## 📊 Social Determinants of Health (SDOH) Analysis

The WHO defines Social Determinants of Health as *"the conditions in which people are born, grow, work, live, and age."*

Let's explore how socioeconomic factors correlate with health outcomes in BC. Key questions:
- Does income predict life expectancy?
- Does poverty drive chronic disease?
- Does access to a family doctor affect ER usage?

In [ ]:
# Income vs life expectancy
fig, ax = plt.subplots(figsize=(12, 7))
for ha, color in ha_colors.items():
    subset = bc_health[bc_health['health_authority'] == ha]
    ax.scatter(subset['median_household_income'], subset['life_expectancy'],
               c=color, label=ha, s=80, alpha=0.7, edgecolors='white', linewidth=0.5)

# Add regression line
x = bc_health['median_household_income'].values
y = bc_health['life_expectancy'].values
slope, intercept, r, p, se = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line, 'k--', alpha=0.5, linewidth=2)
ax.annotate(f'r = {r:.3f}, p = {p:.4f}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Median Household Income ($)')
ax.set_ylabel('Life Expectancy (years)')
ax.set_title('Income vs. Life Expectancy Across BC Communities')
ax.legend(title='Health Authority', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Poverty vs diabetes prevalence
fig, ax = plt.subplots(figsize=(12, 7))
for ha, color in ha_colors.items():
    subset = bc_health[bc_health['health_authority'] == ha]
    ax.scatter(subset['pct_below_poverty_line'], subset['diabetes_prevalence'],
               c=color, label=ha, s=80, alpha=0.7, edgecolors='white', linewidth=0.5)

x = bc_health['pct_below_poverty_line'].values
y = bc_health['diabetes_prevalence'].values
slope, intercept, r, p, se = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line, 'k--', alpha=0.5, linewidth=2)
ax.annotate(f'r = {r:.3f}, p = {p:.4f}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Below Poverty Line (%)')
ax.set_ylabel('Diabetes Prevalence (%)')
ax.set_title('Poverty vs. Diabetes Prevalence Across BC Communities')
ax.legend(title='Health Authority', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Access to family doctor vs ER visits
fig, ax = plt.subplots(figsize=(12, 7))
for ha, color in ha_colors.items():
    subset = bc_health[bc_health['health_authority'] == ha]
    ax.scatter(subset['pct_without_family_doctor'], subset['er_visits_per_1000'],
               c=color, label=ha, s=80, alpha=0.7, edgecolors='white', linewidth=0.5)

x = bc_health['pct_without_family_doctor'].values
y = bc_health['er_visits_per_1000'].values
slope, intercept, r, p, se = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line, 'k--', alpha=0.5, linewidth=2)
ax.annotate(f'r = {r:.3f}, p = {p:.4f}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Without Family Doctor (%)')
ax.set_ylabel('ER Visits per 1,000 Population')
ax.set_title('Access to Primary Care vs. ER Utilization')
ax.legend(title='Health Authority', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Multi-panel SDOH analysis (2x2)
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

panels = [
    ('median_household_income', 'diabetes_prevalence',
     'Median Household Income ($)', 'Diabetes Prevalence (%)'),
    ('pct_below_poverty_line', 'mental_health_hospitalization_rate',
     'Below Poverty Line (%)', 'Mental Health Hospitalizations (per 100K)'),
    ('pct_without_family_doctor', 'er_visits_per_1000',
     'Without Family Doctor (%)', 'ER Visits per 1,000'),
    ('pct_smokers', 'life_expectancy',
     'Smoking Rate (%)', 'Life Expectancy (years)'),
]

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes.flat, panels):
    for ha, color in ha_colors.items():
        subset = bc_health[bc_health['health_authority'] == ha]
        ax.scatter(subset[xcol], subset[ycol], c=color, label=ha,
                   s=50, alpha=0.7, edgecolors='white', linewidth=0.3)

    # Regression line
    x_vals = bc_health[xcol].values
    y_vals = bc_health[ycol].values
    slope, intercept, r, p, se = stats.linregress(x_vals, y_vals)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, intercept + slope * x_line, 'k--', alpha=0.5, linewidth=1.5)
    ax.annotate(f'r = {r:.2f}', xy=(0.05, 0.92), xycoords='axes fraction',
                fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)

axes[0, 0].set_title('Income vs. Diabetes', fontsize=13, fontweight='bold')
axes[0, 1].set_title('Poverty vs. Mental Health', fontsize=13, fontweight='bold')
axes[1, 0].set_title('Doctor Access vs. ER Use', fontsize=13, fontweight='bold')
axes[1, 1].set_title('Smoking vs. Life Expectancy', fontsize=13, fontweight='bold')

# Shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=11,
           title='Health Authority', title_fontsize=12,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Social Determinants of Health \u2192 Health Outcomes in BC',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis — SDOH vs Health Outcomes
sdoh_cols = ['median_household_income', 'pct_below_poverty_line', 'pct_without_family_doctor']
health_cols = ['life_expectancy', 'diabetes_prevalence', 'hypertension_prevalence',
               'mental_health_hospitalization_rate', 'opioid_overdose_rate', 'er_visits_per_1000']

corr_matrix = bc_health[sdoh_cols + health_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix.loc[health_cols, sdoh_cols], annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=1, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation: Social Determinants \u2192 Health Outcomes', fontsize=14, pad=15)
ax.set_xticklabels(['Household\nIncome', 'Below\nPoverty Line', 'Without\nFamily Doctor'],
                    fontsize=11)
ax.set_yticklabels(['Life Expectancy', 'Diabetes', 'Hypertension',
                     'Mental Health\nHospitalization', 'Opioid\nOverdose Rate', 'ER Visits\nper 1000'],
                    fontsize=11, rotation=0)
plt.tight_layout()
plt.show()

---
## ⏱️ Wait Times Across Canada

Wait times for surgical procedures are a critical health system performance metric. 
Let's explore how they vary by province, procedure, and over time — including the COVID-19 impact.

In [ ]:
# Median wait days over time by procedure (national average)
national_avg = wait_times.groupby(['year', 'procedure'])['median_wait_days'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
procedures = national_avg['procedure'].unique()
cmap = plt.cm.tab10
for i, proc in enumerate(procedures):
    proc_data = national_avg[national_avg['procedure'] == proc]
    ax.plot(proc_data['year'], proc_data['median_wait_days'],
            marker='o', markersize=5, label=proc, color=cmap(i), linewidth=2)

# Highlight COVID period
ax.axvspan(2020, 2021, alpha=0.1, color='red', label='COVID-19 impact')
ax.set_xlabel('Year')
ax.set_ylabel('Median Wait Time (days)')
ax.set_title('Canadian Surgical Wait Times by Procedure (National Average)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_xticks(range(2014, 2026))
plt.tight_layout()
plt.show()

In [ ]:
# BC wait times vs national average over time
bc_avg = wait_times[wait_times['province'] == 'BC'].groupby('year')['median_wait_days'].mean()
nat_avg = wait_times.groupby('year')['median_wait_days'].mean()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(nat_avg.index, nat_avg.values, 'b-o', label='National Average', linewidth=2, markersize=7)
ax.plot(bc_avg.index, bc_avg.values, 'r-s', label='British Columbia', linewidth=2.5, markersize=8)
ax.axvspan(2020, 2021, alpha=0.1, color='red')
ax.annotate('COVID-19', xy=(2020.5, ax.get_ylim()[1] * 0.95),
            fontsize=11, ha='center', color='red', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Median Wait Time (days)')
ax.set_title('BC vs. National Average Wait Times')
ax.legend(fontsize=12)
ax.set_xticks(range(2014, 2026))
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: wait times by province and procedure (latest year)
latest_year = wait_times['year'].max()
latest = wait_times[wait_times['year'] == latest_year]
pivot = latest.pivot_table(values='median_wait_days', index='province',
                           columns='procedure', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=1, cbar_kws={'label': 'Median Wait (days)'}, ax=ax)
ax.set_title(f'Median Wait Times by Province and Procedure ({latest_year})', fontsize=14, pad=15)
ax.set_ylabel('Province')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Percentage within benchmark by procedure (latest year, national average)
benchmark_data = latest.groupby('procedure').agg(
    pct_benchmark=('pct_within_benchmark', 'mean'),
    benchmark_days=('benchmark_days', 'first')
).sort_values('pct_benchmark', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bar_colors = ['#F44336' if v < 70 else '#FF9800' if v < 80 else '#4CAF50'
              for v in benchmark_data['pct_benchmark']]
bars = ax.barh(benchmark_data.index, benchmark_data['pct_benchmark'], color=bar_colors)
ax.axvline(x=80, color='black', linestyle='--', alpha=0.5, label='80% target')

for i, (val, (proc, row)) in enumerate(zip(benchmark_data['pct_benchmark'],
                                            benchmark_data.iterrows())):
    ax.text(val + 1, i, f"{val:.0f}% (benchmark: {int(row['benchmark_days'])}d)",
            va='center', fontsize=10)

ax.set_xlabel('Patients Treated Within Benchmark (%)')
ax.set_title(f'Benchmark Compliance by Procedure ({latest_year})')
ax.legend(fontsize=11)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

---
## 💊 The Opioid Crisis in BC

British Columbia has been at the epicenter of Canada's opioid crisis.
In **April 2016**, BC declared a **public health emergency** due to the dramatic increase in opioid overdose deaths.
The crisis worsened during COVID-19 due to toxic drug supply and reduced access to services.

Let's analyze the trajectory.

In [ ]:
# Annual opioid deaths by province
annual_deaths = opioid_data.groupby(['year', 'province'])['apparent_opioid_toxicity_deaths'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
highlight_provinces = ['BC', 'ON', 'AB', 'QC']
other_provinces = [p for p in opioid_data['province'].unique() if p not in highlight_provinces]

# Plot others in grey first
for prov in other_provinces:
    prov_data = annual_deaths[annual_deaths['province'] == prov]
    ax.plot(prov_data['year'], prov_data['apparent_opioid_toxicity_deaths'],
            color='#CCCCCC', linewidth=1, alpha=0.6)

# Highlight key provinces
prov_colors = {'BC': '#F44336', 'ON': '#2196F3', 'AB': '#FF9800', 'QC': '#4CAF50'}
for prov, color in prov_colors.items():
    prov_data = annual_deaths[annual_deaths['province'] == prov]
    lw = 3.5 if prov == 'BC' else 2
    ax.plot(prov_data['year'], prov_data['apparent_opioid_toxicity_deaths'],
            color=color, linewidth=lw, marker='o', markersize=6, label=prov)

ax.set_xlabel('Year')
ax.set_ylabel('Apparent Opioid Toxicity Deaths')
ax.set_title('Opioid Crisis: Annual Deaths by Province')
ax.legend(fontsize=12)
ax.set_xticks(range(2016, 2026))
plt.tight_layout()
plt.show()

In [ ]:
# BC-specific: deaths, hospitalizations, ED visits over time
bc_opioid = opioid_data[opioid_data['province'] == 'BC'].copy()
bc_annual = bc_opioid.groupby('year').agg(
    deaths=('apparent_opioid_toxicity_deaths', 'sum'),
    hospitalizations=('opioid_hospitalizations', 'sum'),
    ed_visits=('opioid_ed_visits', 'sum'),
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 7))
ax2 = ax1.twinx()

ax1.bar(bc_annual['year'] - 0.2, bc_annual['deaths'], width=0.4,
        color='#F44336', alpha=0.8, label='Deaths')
ax1.bar(bc_annual['year'] + 0.2, bc_annual['hospitalizations'], width=0.4,
        color='#FF9800', alpha=0.8, label='Hospitalizations')
ax2.plot(bc_annual['year'], bc_annual['ed_visits'], 'g-s',
         linewidth=2.5, markersize=8, label='ED Visits')

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Deaths / Hospitalizations', fontsize=12, color='#333')
ax2.set_ylabel('ED Visits', fontsize=12, color='green')
ax1.set_title('BC Opioid Crisis: Deaths, Hospitalizations, and ED Visits', fontsize=14)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc='upper left')

ax1.set_xticks(range(2016, 2026))
plt.tight_layout()
plt.show()

In [ ]:
# Death rate per 100K by province (latest year)
latest_opioid_year = opioid_data['year'].max()
latest_opioid = opioid_data[opioid_data['year'] == latest_opioid_year]
prov_rates = latest_opioid.groupby('province')['rate_per_100k_deaths'].mean().sort_values()

fig, ax = plt.subplots(figsize=(12, 7))
bar_colors = ['#F44336' if p == 'BC' else '#64B5F6' for p in prov_rates.index]
ax.barh(prov_rates.index, prov_rates.values, color=bar_colors)
for i, (prov, val) in enumerate(prov_rates.items()):
    ax.text(val + 0.2, i, f'{val:.1f}', va='center', fontsize=10,
            fontweight='bold' if prov == 'BC' else 'normal')

ax.set_xlabel('Opioid Toxicity Death Rate (per 100,000)')
ax.set_title(f'Opioid Death Rate by Province ({latest_opioid_year})')
ax.annotate('BC has the highest rate\nin Canada', xy=(0.75, 0.15),
            xycoords='axes fraction', fontsize=12, color='#F44336',
            fontweight='bold', ha='center',
            bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.8))
plt.tight_layout()
plt.show()

In [ ]:
# BC quarterly trend with key event annotations
bc_quarterly = bc_opioid.copy()
bc_quarterly['period'] = bc_quarterly['year'].astype(str) + '-' + bc_quarterly['quarter']
bc_quarterly = bc_quarterly.sort_values(['year', 'quarter'])
bc_quarterly['period_num'] = range(len(bc_quarterly))

fig, ax = plt.subplots(figsize=(16, 7))
ax.fill_between(bc_quarterly['period_num'], bc_quarterly['apparent_opioid_toxicity_deaths'],
                alpha=0.3, color='#F44336')
ax.plot(bc_quarterly['period_num'], bc_quarterly['apparent_opioid_toxicity_deaths'],
        'r-o', linewidth=2, markersize=4)

# Annotate key events
# Find the index for 2016-Q2 (public health emergency)
phe_idx = bc_quarterly[(bc_quarterly['year'] == 2016) & (bc_quarterly['quarter'] == 'Q2')]['period_num']
if len(phe_idx) > 0:
    phe_x = phe_idx.values[0]
    phe_y = bc_quarterly.loc[phe_idx.index, 'apparent_opioid_toxicity_deaths'].values[0]
    ax.annotate('Public Health\nEmergency Declared\n(Apr 2016)',
                xy=(phe_x, phe_y), xytext=(phe_x + 3, phe_y + phe_y * 0.5),
                fontsize=10, fontweight='bold', color='#B71C1C',
                arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.5),
                bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.8))

# Find 2020-Q2 (COVID impact)
covid_idx = bc_quarterly[(bc_quarterly['year'] == 2020) & (bc_quarterly['quarter'] == 'Q2')]['period_num']
if len(covid_idx) > 0:
    covid_x = covid_idx.values[0]
    covid_y = bc_quarterly.loc[covid_idx.index, 'apparent_opioid_toxicity_deaths'].values[0]
    ax.annotate('COVID-19\nToxic Drug Supply\nCrisis Worsens',
                xy=(covid_x, covid_y), xytext=(covid_x + 3, covid_y + covid_y * 0.3),
                fontsize=10, fontweight='bold', color='#B71C1C',
                arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.5),
                bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.8))

# Set x-ticks to show year labels
year_ticks = bc_quarterly.groupby('year')['period_num'].first()
ax.set_xticks(year_ticks.values)
ax.set_xticklabels(year_ticks.index, fontsize=11)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Apparent Opioid Toxicity Deaths (quarterly)', fontsize=12)
ax.set_title('BC Opioid Crisis: Quarterly Deaths (2016-2025)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🤖 Challenge: Forecast Wait Times

Can we predict future wait times based on historical trends?

We'll build a simple model to forecast BC surgical wait times using:
- Historical wait time data (2014–2025)
- Linear regression with polynomial features
- Compare predictions across procedures

This is a real problem — health system planners need forecasts to allocate surgical capacity.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, r2_score

# Focus on BC knee replacement as our example
bc_knee = wait_times[
    (wait_times['province'] == 'BC') &
    (wait_times['procedure'] == 'Knee Replacement')
].copy().sort_values('year')

print(f"BC Knee Replacement data: {len(bc_knee)} years")
bc_knee[['year', 'median_wait_days', 'pct_within_benchmark', 'benchmark_days', 'volume']]

In [ ]:
# Features and target
X = bc_knee[['year']].values
y = bc_knee['median_wait_days'].values

# Polynomial features to capture non-linear COVID impact
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)

# Train
model = LinearRegression()
model.fit(X_poly, y)
y_pred = model.predict(X_poly)

# Evaluate fit
print(f"Model R\u00b2: {r2_score(y, y_pred):.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y, y_pred):.1f} days")

# Forecast 2026-2028
future_years = np.array([[2026], [2027], [2028]])
future_poly = poly.transform(future_years)
forecast = model.predict(future_poly)

print("\nBC Knee Replacement Wait Time Forecast:")
for yr, days in zip([2026, 2027, 2028], forecast):
    print(f"  {yr}: {days:.0f} days (predicted)")

In [ ]:
# Visualize forecast
fig, ax = plt.subplots(figsize=(13, 7))

# Historical
ax.plot(bc_knee['year'], y, 'bo-', label='Historical', markersize=8, linewidth=2)

# Model fit
ax.plot(bc_knee['year'], y_pred, 'g--', alpha=0.7, label='Model Fit', linewidth=1.5)

# Forecast
ax.plot([2026, 2027, 2028], forecast, 'r^-', markersize=12,
        label='Forecast', linewidth=2.5)

# Benchmark line
benchmark = bc_knee['benchmark_days'].iloc[0]
ax.axhline(y=benchmark, color='orange', linestyle=':', linewidth=2,
           label=f'Benchmark ({benchmark} days)')

# Confidence interval
ax.fill_between([2026, 2027, 2028], forecast * 0.85, forecast * 1.15,
                alpha=0.2, color='red', label='\u00b115% CI')

# COVID annotation
ax.axvspan(2020, 2021, alpha=0.08, color='red')
ax.annotate('COVID-19\nimpact', xy=(2020.5, y.max() * 0.95),
            fontsize=10, ha='center', color='red', fontstyle='italic')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Median Wait (Days)', fontsize=12)
ax.set_title('BC Knee Replacement: Wait Time Forecast', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(range(2014, 2029))
plt.tight_layout()
plt.show()

In [ ]:
# Forecast all BC procedures
print("\n\ud83d\udcca BC Wait Time Forecasts for 2026:")
print("-" * 65)

bc_data = wait_times[wait_times['province'] == 'BC']
forecast_results = []

for procedure in bc_data['procedure'].unique():
    proc_data = bc_data[bc_data['procedure'] == procedure].sort_values('year')
    if len(proc_data) < 5:
        continue

    X_p = proc_data[['year']].values
    y_p = proc_data['median_wait_days'].values

    poly_p = PolynomialFeatures(degree=2)
    X_pp = poly_p.fit_transform(X_p)

    model_p = LinearRegression()
    model_p.fit(X_pp, y_p)

    forecast_2026 = model_p.predict(poly_p.transform([[2026]]))[0]
    benchmark_val = proc_data['benchmark_days'].iloc[0]
    status = "\u2705" if forecast_2026 <= benchmark_val else "\u26a0\ufe0f"

    forecast_results.append({
        'procedure': procedure,
        'forecast_2026': forecast_2026,
        'benchmark': benchmark_val
    })
    print(f"  {status} {procedure:.<40} {forecast_2026:>6.0f} days  (benchmark: {benchmark_val} days)")

print("-" * 65)

In [ ]:
# Visualize all procedure forecasts vs benchmarks
if forecast_results:
    fr_df = pd.DataFrame(forecast_results).sort_values('forecast_2026', ascending=True)

    fig, ax = plt.subplots(figsize=(13, 7))
    y_pos = range(len(fr_df))
    bar_colors = ['#4CAF50' if f <= b else '#F44336'
                  for f, b in zip(fr_df['forecast_2026'], fr_df['benchmark'])]

    ax.barh(y_pos, fr_df['forecast_2026'], color=bar_colors, alpha=0.8, label='Forecast 2026')
    ax.scatter(fr_df['benchmark'], y_pos, color='black', marker='|',
               s=200, linewidths=3, zorder=5, label='Benchmark')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(fr_df['procedure'], fontsize=11)
    ax.set_xlabel('Wait Time (Days)', fontsize=12)
    ax.set_title('BC 2026 Wait Time Forecast vs. Benchmark', fontsize=14)
    ax.legend(fontsize=11)

    # Annotations
    for i, (_, row) in enumerate(fr_df.iterrows()):
        ax.text(max(row['forecast_2026'], row['benchmark']) + 3, i,
                f"{row['forecast_2026']:.0f}d / {int(row['benchmark'])}d",
                va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

---
## 💡 Project Ideas — Where to Go from Here

Now that you have the basics, here are ideas to build a hackathon-winning project:

### Health Equity Dashboard
- Build an interactive map of BC showing health disparities by community
- Use Plotly/Mapbox for geographic visualization
- Highlight communities with the worst SDOH indicators and highest disease burden
- Allow filtering by health authority, age group, or specific condition

### Opioid Crisis Response Tool
- Build a predictive model for overdose hotspots using community-level data
- Combine SDOH variables with opioid surveillance data
- Create a resource allocation optimizer for harm reduction services
- Generate weekly risk scores by community

### Wait Time Optimization
- Build a what-if simulator: *"If we add N surgeons, how do wait times change?"*
- Model patient flow through the surgical pipeline
- Create a patient-facing tool that estimates personalized wait times
- Compare BC to other provinces and identify best practices

### Resource Allocation AI
- Use clustering to identify communities with similar health profiles
- Build a recommendation system for targeted public health interventions
- Optimize hospital resource distribution across Island Health

### Data Sources to Explore Further
- Statistics Canada health tables: https://www.statcan.gc.ca/en/subjects-start/health
- CIHI data tables: https://www.cihi.ca/en/access-data-and-reports/data-tables
- BC Community Health Data: https://communityhealth.phsa.ca
- PHAC open data: https://search.open.canada.ca/opendata?owner_org=phac-aspc

---

**Good luck and happy hacking!** 🚀